In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import requests

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.linear_model import Ridge
from scipy.stats import randint, uniform
from sklearn.model_selection import ParameterSampler

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor


RANDOM_STATE = 42
TARGET = "ClosePrice"

## 1. Load Data

In [2]:
training_set = pd.read_parquet("../data/train_preprocessed.parquet")
testing_set = pd.read_parquet("../data/test_preprocessed.parquet")
df = pd.read_parquet("../data/full_data_preprocessed.parquet")
original_models = pd.read_parquet("../data/baseline_model_performances.parquet")

districts_raw = gpd.read_file("../data/California_School_District_Areas_2024-25.geojson")
districts = districts_raw[districts_raw["DistrictType"].isin(["Unified", "High"])].copy()
districts = districts[["DistrictType", "DistrictName", "geometry"]].to_crs("EPSG:4326")

## 2. Feature Engineering Helpers

In [3]:
raw_lookup = df[["ListingKey", "BedroomsTotal", "BathroomsTotalInteger", "YearBuilt", "Latitude", "Longitude"]]
raw_lookup = raw_lookup.rename(columns={
    "BedroomsTotal": "BedroomsTotal_raw", "BathroomsTotalInteger": "BathroomsTotalInteger_raw",
    "YearBuilt": "YearBuilt_raw", "Latitude": "Latitude_raw", "Longitude": "Longitude_raw",
})
raw_lookup = raw_lookup.drop_duplicates(subset="ListingKey", keep="first")

In [4]:
def merge_raw_lookup(frame, lookup=raw_lookup):
    n_before = len(frame)
    merged = frame.merge(lookup, on="ListingKey", how="left")
    assert len(merged) == n_before, f"Merge changed row count: {n_before} -> {len(merged)}"
    merged.index = frame.index
    return merged

def add_spatial_district(frame, districts):
    points_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(frame["Longitude_raw"], frame["Latitude_raw"]),
        crs="EPSG:4326", index=frame.index,
    )
    joined = gpd.sjoin(points_gdf, districts, how="left", predicate="within")
    joined = joined[~joined.index.duplicated(keep="first")]
    spatial_col = pd.Series(np.nan, index=frame.index, dtype=object)
    spatial_col.loc[joined.index] = joined["DistrictName"].values
    return spatial_col.fillna(frame["HighSchoolDistrict"])

def fetch_arcgis_layer(base_url, where="1=1", out_fields="*", page_size=1000):
    """Paginate through an ArcGIS REST layer's /query endpoint and return a GeoDataFrame."""
    features, offset = [], 0
    while True:
        params = {"where": where, "outFields": out_fields, "f": "geojson",
                   "resultOffset": offset, "resultRecordCount": page_size}
        resp = requests.get(base_url, params=params, timeout=60).json()
        batch = resp.get("features", [])
        if not batch:
            break
        features.extend(batch)
        offset += page_size
        print(f"  fetched {len(features)} features so far...")
        if len(batch) < page_size:
            break
    return gpd.GeoDataFrame.from_features(features)

## 3. Apply Engineered Features

In [5]:
train_df, test_df = training_set.copy(), testing_set.copy()
train_df = merge_raw_lookup(train_df)
test_df = merge_raw_lookup(test_df)

train_df["BedBathRatio"] = train_df["BedroomsTotal_raw"] / train_df["BathroomsTotalInteger_raw"].replace(0, np.nan)
test_df["BedBathRatio"] = test_df["BedroomsTotal_raw"] / test_df["BathroomsTotalInteger_raw"].replace(0, np.nan)
train_med_ratio = train_df["BedBathRatio"].median()
train_df["BedBathRatio"] = train_df["BedBathRatio"].fillna(train_med_ratio)
test_df["BedBathRatio"] = test_df["BedBathRatio"].fillna(train_med_ratio)

train_df["PropertyAgeAtSale"] = (train_df["CloseDate"].dt.year - train_df["YearBuilt_raw"]).clip(lower=0)
test_df["PropertyAgeAtSale"] = (test_df["CloseDate"].dt.year - test_df["YearBuilt_raw"]).clip(lower=0)
train_med_age = train_df["PropertyAgeAtSale"].median()
train_df["PropertyAgeAtSale"] = train_df["PropertyAgeAtSale"].fillna(train_med_age)
test_df["PropertyAgeAtSale"] = test_df["PropertyAgeAtSale"].fillna(train_med_age)

COMP_K = 15
coord_cols = ["Latitude_raw", "Longitude_raw"]
train_coords = train_df[coord_cols].values
train_prices = train_df["ClosePrice"].values

nn = NearestNeighbors(n_neighbors=COMP_K + 1, algorithm="ball_tree").fit(train_coords)
_, neighbor_idx = nn.kneighbors(train_coords)
train_df["CompPriceKNN"] = train_prices[neighbor_idx[:, 1:]].mean(axis=1)

nn_test = NearestNeighbors(n_neighbors=COMP_K, algorithm="ball_tree").fit(train_coords)
_, test_neighbor_idx = nn_test.kneighbors(test_df[coord_cols].values)
test_df["CompPriceKNN"] = train_prices[test_neighbor_idx].mean(axis=1)

train_df["SchoolDistrictSpatial"] = add_spatial_district(train_df, districts)
test_df["SchoolDistrictSpatial"] = add_spatial_district(test_df, districts)

FIRE_LAYER_URL = "https://services.gis.ca.gov/arcgis/rest/services/Environment/Fire_Severity_Zones/MapServer/0/query"
fire_zones_raw = fetch_arcgis_layer(FIRE_LAYER_URL)

LRA_LAYER_URL = "https://services.gis.ca.gov/arcgis/rest/services/Environment/Fire_Severity_Zones/MapServer/1/query"
lra_zones_raw = fetch_arcgis_layer(LRA_LAYER_URL)

FIRE_HAZARD_COL = "HAZ_CLASS"

fire_zones = fire_zones_raw.set_crs("EPSG:4326")[[FIRE_HAZARD_COL, "geometry"]]
lra_zones = lra_zones_raw.set_crs("EPSG:4326")[[FIRE_HAZARD_COL, "geometry"]]

def add_combined_fire_hazard(frame, sra_zones, lra_zones):
    points_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(frame["Longitude_raw"], frame["Latitude_raw"]),
        crs="EPSG:4326", index=frame.index,
    )

    # State Responsibility Area classification
    sra_joined = gpd.sjoin(points_gdf, sra_zones, how="left", predicate="within")
    sra_joined = sra_joined[~sra_joined.index.duplicated(keep="first")]
    result = pd.Series(np.nan, index=frame.index, dtype=object)
    result.loc[sra_joined.index] = sra_joined[FIRE_HAZARD_COL].values

    # for whatever SRA didn't cover, try Local Responsibility Area classification
    still_missing = result.isna()
    if still_missing.any():
        lra_joined = gpd.sjoin(points_gdf.loc[still_missing], lra_zones, how="left", predicate="within")
        lra_joined = lra_joined[~lra_joined.index.duplicated(keep="first")]
        result.loc[lra_joined.index] = lra_joined[FIRE_HAZARD_COL].values

    result = result.str.strip().str.title()

    return result.fillna("Not in Mapped Wildland/Urban Interface Zone") # If outside both

train_df["FireHazardZone"] = add_combined_fire_hazard(train_df, fire_zones, lra_zones)
test_df["FireHazardZone"] = add_combined_fire_hazard(test_df, fire_zones, lra_zones)

  fetched 1000 features so far...
  fetched 2000 features so far...
  fetched 3000 features so far...
  fetched 4000 features so far...
  fetched 5000 features so far...
  fetched 6000 features so far...
  fetched 7000 features so far...
  fetched 8000 features so far...
  fetched 9000 features so far...
  fetched 10000 features so far...
  fetched 11000 features so far...
  fetched 12000 features so far...
  fetched 13000 features so far...
  fetched 14000 features so far...
  fetched 15000 features so far...
  fetched 16000 features so far...
  fetched 17000 features so far...
  fetched 17267 features so far...
  fetched 955 features so far...


## 4. Build Native-Categorical Feature Matrices

In [6]:
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "SchoolDistrictSpatial", "FireHazardZone"]
NON_FEATURE_COLS = ["ListingKey", "CloseDate", "CloseMonth", "HighSchoolDistrict", "Levels",
                     "BedroomsTotal_raw", "BathroomsTotalInteger_raw", "YearBuilt_raw",
                     "Latitude_raw", "Longitude_raw"]

def build_native_X_y(frame):
    feature_cols = [c for c in frame.columns if c not in [TARGET] + NON_FEATURE_COLS]
    X = frame[feature_cols].copy()
    for col in CATEGORICAL_COLS:
        X[col] = X[col].astype("category")

    leftover_object_cols = [c for c in X.columns if X[c].dtype == "object"]
    if leftover_object_cols:
        print(f"WARNING: casting unexpected object-dtype columns to category: {leftover_object_cols}")
        for col in leftover_object_cols:
            X[col] = X[col].astype("category")

    y = frame[TARGET].values
    return X, y

X_train_native, y_train = build_native_X_y(train_df)
X_test_native, y_test = build_native_X_y(test_df)

print(f"{X_train_native.shape[1]} features, {len(CATEGORICAL_COLS)} native categorical (vs. thousands of one-hot columns before)")
print(X_train_native.dtypes.value_counts())

29 features, 5 native categorical (vs. thousands of one-hot columns before)
int64       12
float64     12
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


## 5. Time-Based Fit/Validation Split (for Early Stopping)

In [7]:
sort_idx = train_df["CloseDate"].argsort().values
val_size = int(0.1 * len(train_df))
fit_positions, val_positions = sort_idx[:-val_size], sort_idx[-val_size:]

X_fit, X_val = X_train_native.iloc[fit_positions], X_train_native.iloc[val_positions]
y_fit, y_val = y_train[fit_positions], y_train[val_positions]
print(f"Fit: {len(X_fit)}, internal validation: {len(X_val)}")

Fit: 73672, internal validation: 8185


## 6. Baseline Gradient Boosting Models

### 6a. XGBoost

In [8]:
xgb_model = xgb.XGBRegressor(
    n_estimators=2000, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", enable_categorical=True,
    early_stopping_rounds=50, eval_metric="rmse", random_state=RANDOM_STATE,
)
xgb_model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
print(f"XGBoost best iteration: {xgb_model.best_iteration}")

XGBoost best iteration: 693


### 6b. LightGBM

In [9]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
)
lgb_model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
print(f"LightGBM best iteration: {lgb_model.best_iteration_}")

C:\Users\zahir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM best iteration: 1059


### 6c. CatBoost

In [10]:
cat_model = CatBoostRegressor(
    iterations=2000, learning_rate=0.03, depth=6,
    cat_features=CATEGORICAL_COLS,
    early_stopping_rounds=50, random_state=RANDOM_STATE, verbose=False,
)
cat_model.fit(X_fit, y_fit, eval_set=(X_val, y_val))
print(f"CatBoost best iteration: {cat_model.get_best_iteration()}")

CatBoost best iteration: 1999


## 7. Compare Baseline Boosting Models vs. Original Models

In [11]:
def evaluate_boosting_model(model, name, features_note="Native categorical, early-stopped"):
    test_pred = model.predict(X_test_native)
    return {"Origin": "New", "model": name,
            "test_r2": r2_score(y_test, test_pred), "test_mae": mean_absolute_error(y_test, test_pred),
            "test_rmse": root_mean_squared_error(y_test, test_pred), "test_mape": mean_absolute_percentage_error(y_test, test_pred),
            "New Features": features_note}

boosting_results = pd.DataFrame([
    evaluate_boosting_model(xgb_model, "XGBoost"),
    evaluate_boosting_model(lgb_model, "LightGBM"),
    evaluate_boosting_model(cat_model, "CatBoost"),
]).round(4)

comparison_df = pd.concat([
    original_models.assign(Origin="Old", **{"New Features": ""}),
    boosting_results,
], ignore_index=True)
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,Origin,New Features
0,linear regression,0.8534,0.8420,187946.0976,313321.9877,0.1904,Old,
1,decision tree,0.8418,0.7961,195030.9801,355897.5189,0.1663,Old,
2,random forest,0.9366,0.8734,143843.9497,280387.4237,0.1190,Old,
3,XGBoost,NaN,0.8990,127490.5165,250532.1318,0.1050,New,"Native categorical, early-stopped"
4,LightGBM,NaN,0.9098,122611.9400,236687.0299,0.1012,New,"Native categorical, early-stopped"
5,CatBoost,NaN,0.9037,129275.2875,244529.1273,0.1074,New,"Native categorical, early-stopped"


## 8. Train/Test Gap Check — Baseline Models

In [12]:
def train_test_gap_check(model, name):
    train_pred, test_pred = model.predict(X_train_native), model.predict(X_test_native)
    return {"model": name, "train_r2": r2_score(y_train, train_pred), "test_r2": r2_score(y_test, test_pred)}

gap_check_baseline = pd.DataFrame([
    train_test_gap_check(xgb_model, "XGBoost"),
    train_test_gap_check(lgb_model, "LightGBM"),
    train_test_gap_check(cat_model, "CatBoost"),
])
gap_check_baseline["gap"] = gap_check_baseline["train_r2"] - gap_check_baseline["test_r2"]
gap_check_baseline

,model,train_r2,test_r2,gap
0,XGBoost,0.959061,0.898964,0.060097
1,LightGBM,0.965426,0.909822,0.055604
2,CatBoost,0.929664,0.903747,0.025917


## 9. Hyperparameter Tuning

In [13]:
class _SearchResult:
    """Minimal stand-in for RandomizedSearchCV's result object, so downstream cells
    (gap check, blend, stacking) can keep using `.best_estimator_` / `.best_params_`
    unchanged regardless of which search produced them."""
    def __init__(self, best_estimator_, best_params_, best_val_r2_):
        self.best_estimator_ = best_estimator_
        self.best_params_ = best_params_
        self.best_val_r2_ = best_val_r2_

### 9a. LightGBM

In [14]:
lgb_param_dist = {
    "num_leaves": randint(15, 127), "max_depth": randint(4, 12),
    "learning_rate": uniform(0.01, 0.15),
    "subsample": uniform(0.6, 0.4), "colsample_bytree": uniform(0.6, 0.4),
    "min_child_samples": randint(10, 100),
}
lgb_candidates = list(ParameterSampler(lgb_param_dist, n_iter=25, random_state=RANDOM_STATE))

lgb_results = []
for params in lgb_candidates:
    model = lgb.LGBMRegressor(n_estimators=2000, random_state=RANDOM_STATE, verbose=-1, **params)
    model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    val_r2 = r2_score(y_val, model.predict(X_val))
    lgb_results.append((params, val_r2, model))

lgb_best_params, lgb_best_val_r2, lgb_best_model = max(lgb_results, key=lambda r: r[1])
print(f"Best LightGBM params: {lgb_best_params} | validation R2: {lgb_best_val_r2:.4f}")

lgb_search = _SearchResult(lgb_best_model, lgb_best_params, lgb_best_val_r2)

comparison_df = pd.concat([comparison_df, pd.DataFrame([
    evaluate_boosting_model(lgb_search.best_estimator_, "LightGBM (tuned)", "Manual randomized search, chronological val split, early-stopped")
])], ignore_index=True)
comparison_df

C:\Users\zahir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\zahir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\zahir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set

Best LightGBM params: {'colsample_bytree': np.float64(0.6421977039321082), 'learning_rate': np.float64(0.07848018557243652), 'max_depth': 7, 'min_child_samples': 53, 'num_leaves': 98, 'subsample': np.float64(0.9570235993959911)} | validation R2: 0.9089


,model,train_r2,test_r2,test_mae,test_rmse,test_mape,Origin,New Features
0,linear regression,0.8534,0.842000,187946.097600,313321.987700,0.190400,Old,
1,decision tree,0.8418,0.796100,195030.980100,355897.518900,0.166300,Old,
2,random forest,0.9366,0.873400,143843.949700,280387.423700,0.119000,Old,
3,XGBoost,NaN,0.899000,127490.516500,250532.131800,0.105000,New,"Native categorical, early-stopped"
4,LightGBM,NaN,0.909800,122611.940000,236687.029900,0.101200,New,"Native categorical, early-stopped"
5,CatBoost,NaN,0.903700,129275.287500,244529.127300,0.107400,New,"Native categorical, early-stopped"
6,LightGBM (tuned),NaN,0.907592,123579.984904,239595.578752,0.101529,New,"Manual randomized search, chronological val sp..."


### 9b. XGBoost

In [15]:
xgb_param_dist = {
    "max_depth": randint(3, 10), "learning_rate": uniform(0.01, 0.15),
    "subsample": uniform(0.6, 0.4), "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
}
xgb_candidates = list(ParameterSampler(xgb_param_dist, n_iter=25, random_state=RANDOM_STATE))

xgb_results = []
for params in xgb_candidates:
    model = xgb.XGBRegressor(
        n_estimators=2000, tree_method="hist", enable_categorical=True,
        early_stopping_rounds=50, eval_metric="rmse", random_state=RANDOM_STATE, **params,
    )
    model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
    val_r2 = r2_score(y_val, model.predict(X_val))
    xgb_results.append((params, val_r2, model))

xgb_best_params, xgb_best_val_r2, xgb_best_model = max(xgb_results, key=lambda r: r[1])
print(f"Best XGBoost params: {xgb_best_params} | validation R2: {xgb_best_val_r2:.4f}")

xgb_search = _SearchResult(xgb_best_model, xgb_best_params, xgb_best_val_r2)

comparison_df = pd.concat([comparison_df, pd.DataFrame([
    evaluate_boosting_model(xgb_search.best_estimator_, "XGBoost (tuned)", "Manual randomized search, chronological val split, early-stopped")
])], ignore_index=True)
comparison_df

Best XGBoost params: {'colsample_bytree': np.float64(0.6063865008880857), 'learning_rate': np.float64(0.04463407384332235), 'max_depth': 6, 'min_child_weight': 7, 'subsample': np.float64(0.8439986631130484)} | validation R2: 0.9082


,model,train_r2,test_r2,test_mae,test_rmse,test_mape,Origin,New Features
0,linear regression,0.8534,0.842000,187946.097600,313321.987700,0.190400,Old,
1,decision tree,0.8418,0.796100,195030.980100,355897.518900,0.166300,Old,
2,random forest,0.9366,0.873400,143843.949700,280387.423700,0.119000,Old,
3,XGBoost,NaN,0.899000,127490.516500,250532.131800,0.105000,New,"Native categorical, early-stopped"
4,LightGBM,NaN,0.909800,122611.940000,236687.029900,0.101200,New,"Native categorical, early-stopped"
5,CatBoost,NaN,0.903700,129275.287500,244529.127300,0.107400,New,"Native categorical, early-stopped"
6,LightGBM (tuned),NaN,0.907592,123579.984904,239595.578752,0.101529,New,"Manual randomized search, chronological val sp..."
7,XGBoost (tuned),NaN,0.910114,123092.266199,236303.381511,0.103092,New,"Manual randomized search, chronological val sp..."


### 9c. CatBoost

In [16]:
cat_param_dist = {
    "depth": randint(4, 6), "learning_rate": uniform(0.01, 0.15), "l2_leaf_reg": uniform(3, 9),
}
cat_candidates = list(ParameterSampler(cat_param_dist, n_iter=10, random_state=RANDOM_STATE))

cat_results = []
for i, params in enumerate(cat_candidates, 1):
    print(f"CatBoost candidate {i}/{len(cat_candidates)}: {params}")
    model = CatBoostRegressor(
        iterations=2000, cat_features=CATEGORICAL_COLS, random_state=RANDOM_STATE,
        verbose=False, early_stopping_rounds=50, thread_count=-1,
        max_ctr_complexity=1, 
        **params,
    )
    model.fit(X_fit, y_fit, eval_set=(X_val, y_val))
    val_r2 = r2_score(y_val, model.predict(X_val))
    cat_results.append((params, val_r2, model))

cat_best_params, cat_best_val_r2, cat_best_model = max(cat_results, key=lambda r: r[1])
print(f"Best CatBoost params: {cat_best_params} | validation R2: {cat_best_val_r2:.4f}")

cat_search = _SearchResult(cat_best_model, cat_best_params, cat_best_val_r2)

comparison_df = pd.concat([comparison_df, pd.DataFrame([
    evaluate_boosting_model(cat_search.best_estimator_, "CatBoost (tuned)", "Manual randomized search, chronological val split, early-stopped, max_ctr_complexity=1")
])], ignore_index=True)

CatBoost candidate 1/10: {'depth': 4, 'l2_leaf_reg': np.float64(10.168886881742097), 'learning_rate': np.float64(0.03751521847992457)}
CatBoost candidate 2/10: {'depth': 5, 'l2_leaf_reg': np.float64(8.38792635777333), 'learning_rate': np.float64(0.033402796066365474)}
CatBoost candidate 3/10: {'depth': 4, 'l2_leaf_reg': np.float64(3.899774242362026), 'learning_rate': np.float64(0.07888733379488007)}
CatBoost candidate 4/10: {'depth': 4, 'l2_leaf_reg': np.float64(8.41003510568888), 'learning_rate': np.float64(0.11621088666940682)}
CatBoost candidate 5/10: {'depth': 5, 'l2_leaf_reg': np.float64(3.5077042112439023), 'learning_rate': np.float64(0.1182998158400237)}
CatBoost candidate 6/10: {'depth': 5, 'l2_leaf_reg': np.float64(4.911051996104486), 'learning_rate': np.float64(0.03727374508106509)}
CatBoost candidate 7/10: {'depth': 4, 'l2_leaf_reg': np.float64(8.557333586649449), 'learning_rate': np.float64(0.10174797407324213)}
CatBoost candidate 8/10: {'depth': 4, 'l2_leaf_reg': np.float6

In [17]:
comparison_df = comparison_df.drop_duplicates(subset="model", keep="last").reset_index(drop=True)
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,Origin,New Features
0,linear regression,0.8534,0.842000,187946.097600,313321.987700,0.190400,Old,
1,decision tree,0.8418,0.796100,195030.980100,355897.518900,0.166300,Old,
2,random forest,0.9366,0.873400,143843.949700,280387.423700,0.119000,Old,
3,XGBoost,NaN,0.899000,127490.516500,250532.131800,0.105000,New,"Native categorical, early-stopped"
4,LightGBM,NaN,0.909800,122611.940000,236687.029900,0.101200,New,"Native categorical, early-stopped"
5,CatBoost,NaN,0.903700,129275.287500,244529.127300,0.107400,New,"Native categorical, early-stopped"
6,LightGBM (tuned),NaN,0.907592,123579.984904,239595.578752,0.101529,New,"Manual randomized search, chronological val sp..."
7,XGBoost (tuned),NaN,0.910114,123092.266199,236303.381511,0.103092,New,"Manual randomized search, chronological val sp..."
8,CatBoost (tuned),NaN,0.906226,126570.069847,241360.064144,0.104965,New,"Manual randomized search, chronological val sp..."


## 10. Train/Test Gap Check — Tuned Models

In [18]:
gap_check_tuned = pd.DataFrame([
    train_test_gap_check(xgb_search.best_estimator_, "XGBoost (tuned)"),
    train_test_gap_check(lgb_search.best_estimator_, "LightGBM (tuned)"),
    train_test_gap_check(cat_search.best_estimator_, "CatBoost (tuned)"),
])
gap_check_tuned["gap"] = gap_check_tuned["train_r2"] - gap_check_tuned["test_r2"]
gap_check_tuned

,model,train_r2,test_r2,gap
0,XGBoost (tuned),0.963179,0.910114,0.053065
1,LightGBM (tuned),0.963041,0.907592,0.055449
2,CatBoost (tuned),0.935852,0.906226,0.029625


## 11. Feature Importance Analysis

In [19]:
def get_gain_importance(model, feature_names, kind):
    if kind == "xgb":
        raw = model.get_booster().get_score(importance_type="gain")
    elif kind == "lgb":
        raw = dict(zip(model.booster_.feature_name(), model.booster_.feature_importance(importance_type="gain")))
    elif kind == "cat":
        raw = dict(zip(model.feature_names_, model.get_feature_importance()))
    importance = np.array([raw.get(f, 0.0) for f in feature_names], dtype=float)
    total = importance.sum()
    return importance / total if total > 0 else importance

feature_names = list(X_train_native.columns)
importance_df = pd.DataFrame({
    "feature": feature_names,
    "xgb_importance": get_gain_importance(xgb_search.best_estimator_, feature_names, "xgb"),
    "lgb_importance": get_gain_importance(lgb_search.best_estimator_, feature_names, "lgb"),
    "cat_importance": get_gain_importance(cat_search.best_estimator_, feature_names, "cat"),
})
importance_df["mean_importance"] = importance_df[["xgb_importance", "lgb_importance", "cat_importance"]].mean(axis=1)
importance_df = importance_df.sort_values("mean_importance", ascending=False).reset_index(drop=True)
importance_df

,feature,xgb_importance,lgb_importance,cat_importance,mean_importance
0,CompPriceKNN,0.334225,0.599494,0.531466,0.488395
1,LivingArea,0.072345,0.126309,0.159420,0.119358
2,City,0.172575,0.020613,0.009991,0.067726
3,BathroomsTotalInteger,0.088265,0.045359,0.063606,0.065743
4,SchoolDistrictSpatial,0.084454,0.067431,0.033611,0.061832
5,MLSAreaMajor,0.037800,0.040957,0.014355,0.031037
6,PostalCode,0.043520,0.032310,0.012128,0.029319
7,Longitude,0.007098,0.006526,0.042199,0.018608
8,Latitude,0.005719,0.005877,0.039206,0.016934
9,LotSizeSquareFeet,0.009484,0.011099,0.019408,0.013330


## 12. Feature Pruning Experiment

In [20]:
PRUNE_THRESHOLD = 0.0015  # tested extensively and found this value to perform best
features_to_drop = importance_df.loc[importance_df["mean_importance"] < PRUNE_THRESHOLD, "feature"].tolist()
dropped_share = importance_df.loc[importance_df["feature"].isin(features_to_drop), "mean_importance"].sum()
print(f"Dropping {len(features_to_drop)} features ({dropped_share:.1%} of total mean importance): {features_to_drop}")

X_train_pruned = X_train_native.drop(columns=features_to_drop)
X_test_pruned = X_test_native.drop(columns=features_to_drop)
X_fit_pruned, X_val_pruned = X_train_pruned.iloc[fit_positions], X_train_pruned.iloc[val_positions]
PRUNED_CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c not in features_to_drop]

def evaluate_pruned(model, name, X_eval):
    test_pred = model.predict(X_eval)
    return {"Origin": "New", "model": name,
            "test_r2": r2_score(y_test, test_pred), "test_mae": mean_absolute_error(y_test, test_pred),
            "test_rmse": root_mean_squared_error(y_test, test_pred), "test_mape": mean_absolute_percentage_error(y_test, test_pred),
            "New Features": f"{len(features_to_drop)} low-importance features dropped, same tuned hyperparameters"}

xgb_pruned = xgb.XGBRegressor(
    n_estimators=2000, tree_method="hist", enable_categorical=True,
    early_stopping_rounds=50, eval_metric="rmse", random_state=RANDOM_STATE, **xgb_search.best_params_,
)
xgb_pruned.fit(X_fit_pruned, y_fit, eval_set=[(X_val_pruned, y_val)], verbose=False)

lgb_pruned = lgb.LGBMRegressor(n_estimators=2000, random_state=RANDOM_STATE, verbose=-1, **lgb_search.best_params_)
lgb_pruned.fit(X_fit_pruned, y_fit, eval_set=[(X_val_pruned, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])

cat_pruned = CatBoostRegressor(
    iterations=2000, cat_features=PRUNED_CATEGORICAL_COLS, random_state=RANDOM_STATE,
    verbose=False, early_stopping_rounds=50, thread_count=-1, max_ctr_complexity=1, **cat_search.best_params_,
)
cat_pruned.fit(X_fit_pruned, y_fit, eval_set=(X_val_pruned, y_val))

pruned_results = pd.DataFrame([
    evaluate_pruned(xgb_pruned, "XGBoost (pruned)", X_test_pruned),
    evaluate_pruned(lgb_pruned, "LightGBM (pruned)", X_test_pruned),
    evaluate_pruned(cat_pruned, "CatBoost (pruned)", X_test_pruned),
])
comparison_df = pd.concat([comparison_df, pruned_results], ignore_index=True)

print(f"Pruned Threshold: {PRUNE_THRESHOLD}")
print("\nPruned vs. full-feature tuned models (test R2):")
for full_name, pruned_name in [("XGBoost (tuned)", "XGBoost (pruned)"), ("LightGBM (tuned)", "LightGBM (pruned)"), ("CatBoost (tuned)", "CatBoost (pruned)")]:
    full_r2 = comparison_df.loc[comparison_df["model"] == full_name, "test_r2"].iloc[-1]
    pruned_r2 = comparison_df.loc[comparison_df["model"] == pruned_name, "test_r2"].iloc[-1]
    print(f"  {full_name}: {full_r2:.4f}  ->  {pruned_name}: {pruned_r2:.4f}  (Δ {pruned_r2 - full_r2:+.4f})")

pruned_results

Dropping 3 features (0.4% of total mean importance): ['AttachedGarageYN', 'Level_MultiSplit', 'BasementYN']


C:\Users\zahir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Pruned Threshold: 0.0015

Pruned vs. full-feature tuned models (test R2):
  XGBoost (tuned): 0.9101  ->  XGBoost (pruned): 0.9100  (Δ -0.0001)
  LightGBM (tuned): 0.9076  ->  LightGBM (pruned): 0.9102  (Δ +0.0026)
  CatBoost (tuned): 0.9062  ->  CatBoost (pruned): 0.9066  (Δ +0.0004)


,Origin,model,test_r2,test_mae,test_rmse,test_mape,New Features
0,New,XGBoost (pruned),0.910019,122650.112645,236428.651748,0.103194,"3 low-importance features dropped, same tuned ..."
1,New,LightGBM (pruned),0.910218,122928.155170,236167.536925,0.102200,"3 low-importance features dropped, same tuned ..."
2,New,CatBoost (pruned),0.906632,126489.114515,240837.252170,0.104928,"3 low-importance features dropped, same tuned ..."


## 13. Ensemble Blend of Tuned Models (Weighted Average)

In [21]:
xgb_val_pred = xgb_search.best_estimator_.predict(X_val)
lgb_val_pred = lgb_search.best_estimator_.predict(X_val)
cat_val_pred = cat_search.best_estimator_.predict(X_val)

best_r2, best_weights = -np.inf, None
for w_xgb in np.arange(0, 1.01, 0.05):
    for w_lgb in np.arange(0, 1.01 - w_xgb, 0.05):
        w_cat = 1 - w_xgb - w_lgb
        if w_cat < -1e-9:
            continue
        blend_val = w_xgb * xgb_val_pred + w_lgb * lgb_val_pred + w_cat * cat_val_pred
        r2 = r2_score(y_val, blend_val)
        if r2 > best_r2:
            best_r2, best_weights = r2, (round(w_xgb, 2), round(w_lgb, 2), round(w_cat, 2))

print(f"Best weights found on validation (xgb, lgb, cat): {best_weights}, validation R2: {best_r2:.4f}")

w_xgb, w_lgb, w_cat = best_weights
blend_test_pred = (w_xgb * xgb_search.best_estimator_.predict(X_test_native) +
                    w_lgb * lgb_search.best_estimator_.predict(X_test_native) +
                    w_cat * cat_search.best_estimator_.predict(X_test_native))

comparison_df = pd.concat([comparison_df, pd.DataFrame([{
    "Origin": "New", "model": "XGB+LGB+CatBoost blend (tuned)", "test_r2": r2_score(y_test, blend_test_pred),
    "test_mae": mean_absolute_error(y_test, blend_test_pred), "test_rmse": root_mean_squared_error(y_test, blend_test_pred),
    "test_mape": mean_absolute_percentage_error(y_test, blend_test_pred), "New Features": f"Weighted blend {best_weights} of tuned models",
}])], ignore_index=True)
comparison_df

Best weights found on validation (xgb, lgb, cat): (np.float64(0.25), np.float64(0.4), np.float64(0.35)), validation R2: 0.9119


,model,train_r2,test_r2,test_mae,test_rmse,test_mape,Origin,New Features
0,linear regression,0.8534,0.842000,187946.097600,313321.987700,0.190400,Old,
1,decision tree,0.8418,0.796100,195030.980100,355897.518900,0.166300,Old,
2,random forest,0.9366,0.873400,143843.949700,280387.423700,0.119000,Old,
3,XGBoost,NaN,0.899000,127490.516500,250532.131800,0.105000,New,"Native categorical, early-stopped"
4,LightGBM,NaN,0.909800,122611.940000,236687.029900,0.101200,New,"Native categorical, early-stopped"
5,CatBoost,NaN,0.903700,129275.287500,244529.127300,0.107400,New,"Native categorical, early-stopped"
6,LightGBM (tuned),NaN,0.907592,123579.984904,239595.578752,0.101529,New,"Manual randomized search, chronological val sp..."
7,XGBoost (tuned),NaN,0.910114,123092.266199,236303.381511,0.103092,New,"Manual randomized search, chronological val sp..."
8,CatBoost (tuned),NaN,0.906226,126570.069847,241360.064144,0.104965,New,"Manual randomized search, chronological val sp..."
9,XGBoost (pruned),NaN,0.910019,122650.112645,236428.651748,0.103194,New,"3 low-importance features dropped, same tuned ..."


## 14. Stacked Ensemble (Ridge Meta-Learner)

In [22]:
stack_X_val = np.column_stack([xgb_val_pred, lgb_val_pred, cat_val_pred])
stack_X_test = np.column_stack([
    xgb_search.best_estimator_.predict(X_test_native),
    lgb_search.best_estimator_.predict(X_test_native),
    cat_search.best_estimator_.predict(X_test_native),
])

meta_learner = Ridge(alpha=1.0, random_state=RANDOM_STATE)
meta_learner.fit(stack_X_val, y_val)
stack_test_pred = meta_learner.predict(stack_X_test)

print(f"Stacked (Ridge) test R2: {r2_score(y_test, stack_test_pred):.4f}")
print(f"Meta-learner coefficients (xgb, lgb, cat): {meta_learner.coef_.round(3)}, intercept: {meta_learner.intercept_:.1f}")

comparison_df = pd.concat([comparison_df, pd.DataFrame([{
    "Origin": "New", "model": "XGB+LGB+CatBoost stacked (Ridge)", "test_r2": r2_score(y_test, stack_test_pred),
    "test_mae": mean_absolute_error(y_test, stack_test_pred), "test_rmse": root_mean_squared_error(y_test, stack_test_pred),
    "test_mape": mean_absolute_percentage_error(y_test, stack_test_pred), "New Features": "Ridge meta-learner on tuned model predictions",
}])], ignore_index=True)
comparison_df

Stacked (Ridge) test R2: 0.9137
Meta-learner coefficients (xgb, lgb, cat): [0.274 0.376 0.374], intercept: -6720.2


,model,train_r2,test_r2,test_mae,test_rmse,test_mape,Origin,New Features
0,linear regression,0.8534,0.842000,187946.097600,313321.987700,0.190400,Old,
1,decision tree,0.8418,0.796100,195030.980100,355897.518900,0.166300,Old,
2,random forest,0.9366,0.873400,143843.949700,280387.423700,0.119000,Old,
3,XGBoost,NaN,0.899000,127490.516500,250532.131800,0.105000,New,"Native categorical, early-stopped"
4,LightGBM,NaN,0.909800,122611.940000,236687.029900,0.101200,New,"Native categorical, early-stopped"
5,CatBoost,NaN,0.903700,129275.287500,244529.127300,0.107400,New,"Native categorical, early-stopped"
6,LightGBM (tuned),NaN,0.907592,123579.984904,239595.578752,0.101529,New,"Manual randomized search, chronological val sp..."
7,XGBoost (tuned),NaN,0.910114,123092.266199,236303.381511,0.103092,New,"Manual randomized search, chronological val sp..."
8,CatBoost (tuned),NaN,0.906226,126570.069847,241360.064144,0.104965,New,"Manual randomized search, chronological val sp..."
9,XGBoost (pruned),NaN,0.910019,122650.112645,236428.651748,0.103194,New,"3 low-importance features dropped, same tuned ..."


## 15. Pruned Ensemble (Blend + Stack)

In [23]:
xgb_val_pred_pruned = xgb_pruned.predict(X_val_pruned)
lgb_val_pred_pruned = lgb_pruned.predict(X_val_pruned)
cat_val_pred_pruned = cat_pruned.predict(X_val_pruned)

best_r2_pruned, best_weights_pruned = -np.inf, None
for w_xgb in np.arange(0, 1.01, 0.05):
    for w_lgb in np.arange(0, 1.01 - w_xgb, 0.05):
        w_cat = 1 - w_xgb - w_lgb
        if w_cat < -1e-9:
            continue
        blend_val = w_xgb * xgb_val_pred_pruned + w_lgb * lgb_val_pred_pruned + w_cat * cat_val_pred_pruned
        r2 = r2_score(y_val, blend_val)
        if r2 > best_r2_pruned:
            best_r2_pruned, best_weights_pruned = r2, (round(w_xgb, 2), round(w_lgb, 2), round(w_cat, 2))

print(f"Best pruned-blend weights (xgb, lgb, cat): {best_weights_pruned}, validation R2: {best_r2_pruned:.4f}")

w_xgb, w_lgb, w_cat = best_weights_pruned
blend_test_pred_pruned = (w_xgb * xgb_pruned.predict(X_test_pruned) +
                           w_lgb * lgb_pruned.predict(X_test_pruned) +
                           w_cat * cat_pruned.predict(X_test_pruned))

stack_X_val_pruned = np.column_stack([xgb_val_pred_pruned, lgb_val_pred_pruned, cat_val_pred_pruned])
stack_X_test_pruned = np.column_stack([xgb_pruned.predict(X_test_pruned), lgb_pruned.predict(X_test_pruned), cat_pruned.predict(X_test_pruned)])

meta_learner_pruned = Ridge(alpha=1.0, random_state=RANDOM_STATE)
meta_learner_pruned.fit(stack_X_val_pruned, y_val)
stack_test_pred_pruned = meta_learner_pruned.predict(stack_X_test_pruned)
print(f"Pruned stacked (Ridge) test R2: {r2_score(y_test, stack_test_pred_pruned):.4f}")
print(f"Meta-learner coefficients (xgb, lgb, cat): {meta_learner_pruned.coef_.round(3)}, intercept: {meta_learner_pruned.intercept_:.1f}")

pruned_ensemble_results = pd.DataFrame([
    {"Origin": "New", "model": "Pruned: XGB+LGB+CatBoost blend (tuned)", "test_r2": r2_score(y_test, blend_test_pred_pruned),
     "test_mae": mean_absolute_error(y_test, blend_test_pred_pruned), "test_rmse": root_mean_squared_error(y_test, blend_test_pred_pruned),
     "test_mape": mean_absolute_percentage_error(y_test, blend_test_pred_pruned), "New Features": f"Pruned features, weighted blend {best_weights_pruned}"},
    {"Origin": "New", "model": "Pruned: XGB+LGB+CatBoost stacked (Ridge)", "test_r2": r2_score(y_test, stack_test_pred_pruned),
     "test_mae": mean_absolute_error(y_test, stack_test_pred_pruned), "test_rmse": root_mean_squared_error(y_test, stack_test_pred_pruned),
     "test_mape": mean_absolute_percentage_error(y_test, stack_test_pred_pruned), "New Features": "Pruned features, Ridge meta-learner"},
])
comparison_df = pd.concat([comparison_df, pruned_ensemble_results], ignore_index=True)

print("\nPruned vs. full-feature ensemble (test R2):")
for full_name, pruned_name in [("XGB+LGB+CatBoost blend (tuned)", "Pruned: XGB+LGB+CatBoost blend (tuned)"),
                                ("XGB+LGB+CatBoost stacked (Ridge)", "Pruned: XGB+LGB+CatBoost stacked (Ridge)")]:
    full_r2 = comparison_df.loc[comparison_df["model"] == full_name, "test_r2"].iloc[-1]
    pruned_r2 = comparison_df.loc[comparison_df["model"] == pruned_name, "test_r2"].iloc[-1]
    print(f"  {full_name}: {full_r2:.4f}  ->  {pruned_name}: {pruned_r2:.4f}  (Δ {pruned_r2 - full_r2:+.4f})")

pruned_ensemble_results

Best pruned-blend weights (xgb, lgb, cat): (np.float64(0.35), np.float64(0.3), np.float64(0.35)), validation R2: 0.9109
Pruned stacked (Ridge) test R2: 0.9145
Meta-learner coefficients (xgb, lgb, cat): [0.374 0.256 0.395], intercept: -8102.7

Pruned vs. full-feature ensemble (test R2):
  XGB+LGB+CatBoost blend (tuned): 0.9121  ->  Pruned: XGB+LGB+CatBoost blend (tuned): 0.9130  (Δ +0.0008)
  XGB+LGB+CatBoost stacked (Ridge): 0.9137  ->  Pruned: XGB+LGB+CatBoost stacked (Ridge): 0.9145  (Δ +0.0007)


,Origin,model,test_r2,test_mae,test_rmse,test_mape,New Features
0,New,Pruned: XGB+LGB+CatBoost blend (tuned),0.912963,120444.518271,232529.235599,0.100070,"Pruned features, weighted blend (np.float64(0...."
1,New,Pruned: XGB+LGB+CatBoost stacked (Ridge),0.914478,119751.520676,230495.842084,0.101024,"Pruned features, Ridge meta-learner"


## 16. Final Comparison & Export

In [24]:
comparison_df = comparison_df.drop_duplicates(subset="model", keep="last").reset_index(drop=True)
comparison_df = comparison_df.sort_values("test_r2", ascending=False).reset_index(drop=True)
print(f"Best model: {comparison_df.iloc[0]['model']} — test R2 = {comparison_df.iloc[0]['test_r2']:.4f}")
comparison_df

Best model: Pruned: XGB+LGB+CatBoost stacked (Ridge) — test R2 = 0.9145


,model,train_r2,test_r2,test_mae,test_rmse,test_mape,Origin,New Features
0,Pruned: XGB+LGB+CatBoost stacked (Ridge),NaN,0.914478,119751.520676,230495.842084,0.101024,New,"Pruned features, Ridge meta-learner"
1,XGB+LGB+CatBoost stacked (Ridge),NaN,0.913736,119590.381402,231493.908825,0.100089,New,Ridge meta-learner on tuned model predictions
2,Pruned: XGB+LGB+CatBoost blend (tuned),NaN,0.912963,120444.518271,232529.235599,0.100070,New,"Pruned features, weighted blend (np.float64(0...."
3,XGB+LGB+CatBoost blend (tuned),NaN,0.912128,120515.106816,233641.587245,0.099282,New,"Weighted blend (np.float64(0.25), np.float64(0..."
4,LightGBM (pruned),NaN,0.910218,122928.155170,236167.536925,0.102200,New,"3 low-importance features dropped, same tuned ..."
5,XGBoost (tuned),NaN,0.910114,123092.266199,236303.381511,0.103092,New,"Manual randomized search, chronological val sp..."
6,XGBoost (pruned),NaN,0.910019,122650.112645,236428.651748,0.103194,New,"3 low-importance features dropped, same tuned ..."
7,LightGBM,NaN,0.909800,122611.940000,236687.029900,0.101200,New,"Native categorical, early-stopped"
8,LightGBM (tuned),NaN,0.907592,123579.984904,239595.578752,0.101529,New,"Manual randomized search, chronological val sp..."
9,CatBoost (pruned),NaN,0.906632,126489.114515,240837.252170,0.104928,New,"3 low-importance features dropped, same tuned ..."


In [25]:
comparison_df.to_parquet("../data/advanced_model_performances.parquet", index=False)

test_predictions = pd.DataFrame({
    "ListingKey": test_df["ListingKey"].values,
    "CloseDate": test_df["CloseDate"].values,
    "ClosePrice": y_test,
    "PredictedClosePrice": stack_test_pred_pruned,
})
test_predictions.to_parquet("../data/test_predictions.parquet", index=False)
print(f"Saved {len(test_predictions)} test predictions from the final model "
      f"(Pruned: XGB+LGB+CatBoost stacked (Ridge), test R2 = {r2_score(y_test, stack_test_pred_pruned):.4f})")
test_predictions.head()

Saved 7442 test predictions from the final model (Pruned: XGB+LGB+CatBoost stacked (Ridge), test R2 = 0.9145)


,ListingKey,CloseDate,ClosePrice,PredictedClosePrice
0,1171562218,2026-05-28,1625000.0,1.437726e+06
1,1171541127,2026-05-15,1560000.0,1.672927e+06
2,1171287279,2026-05-29,377500.0,3.646478e+05
3,1171065918,2026-05-29,595000.0,8.554073e+05
4,1170805462,2026-05-29,1890000.0,2.508648e+06
